# ECG Arrhythmia Detection — 1D CNN Prototype
**Google Colab | MIT-BIH Arrhythmia Database | Python**

This notebook trains a compact 1D CNN, then calculates metrics on held-out ECG records. Run cells from top to bottom. Results are computed by the notebook; no scores are pre-filled or invented.

The record-level DS1/DS2 split helps avoid leakage between individual beats from the same recording. This is an educational prototype, not a medical diagnostic system.


## 1. Install and import

In [ ]:
!pip -q install wfdb

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import tensorflow as tf
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


## 2. Download MIT-BIH records
The following commonly used DS1/DS2 record groups are used for a record-level train/test split.

In [ ]:
DATA_DIR = "/content/mitdb"
os.makedirs(DATA_DIR, exist_ok=True)
DS1 = ["101","106","108","109","112","114","115","116","118","119","122","124",
       "201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210","212",
       "213","214","219","221","222","228","231","232","233","234"]
RECORDS = DS1 + DS2
wfdb.dl_database("mitdb", dl_dir=DATA_DIR, records=RECORDS)
print("Downloaded", len(RECORDS), "records")


## 3. Preprocessing and label mapping
Each annotated beat is cut into a fixed-length segment. Annotation symbols are grouped into five broad AAMI-style classes: N, S, V, F, Q. This is a simplified prototype mapping; the printed class distribution and per-class report are important when interpreting results.

In [ ]:
CLASS_MAP = {
    "N":"N", "L":"N", "R":"N", "e":"N", "j":"N",
    "A":"S", "a":"S", "J":"S", "S":"S",
    "V":"V", "E":"V", "F":"F",
    "/":"Q", "f":"Q", "Q":"Q"
}
CLASSES = ["N", "S", "V", "F", "Q"]
TO_ID = {c:i for i,c in enumerate(CLASSES)}
FS = 360
BEFORE, AFTER = int(.20*FS), int(.40*FS)
SEG_LEN = BEFORE + AFTER

def filter_ecg(x, fs=FS):
    nyq = fs/2
    b, a = butter(3, [0.5/nyq, 40/nyq], btype="band")
    return filtfilt(b, a, x) if len(x) > 3*max(len(a),len(b)) else x

def normalize(x):
    x = np.asarray(x, dtype=np.float32)
    sd = x.std()
    if sd < 1e-6: return None
    return (x-x.mean())/sd

def read_beats(rec_name):
    path = os.path.join(DATA_DIR, rec_name)
    rec = wfdb.rdrecord(path, channels=[0])
    ann = wfdb.rdann(path, "atr")
    raw = rec.p_signal[:,0].astype(np.float32)
    clean = filter_ecg(raw, int(rec.fs))
    X, y = [], []
    for sample, symbol in zip(ann.sample, ann.symbol):
        if symbol not in CLASS_MAP: continue
        start, end = int(sample)-BEFORE, int(sample)+AFTER
        if start < 0 or end > len(clean): continue
        beat = normalize(clean[start:end])
        if beat is not None and len(beat)==SEG_LEN:
            X.append(beat); y.append(TO_ID[CLASS_MAP[symbol]])
    return raw, clean, np.asarray(X, np.float32), np.asarray(y, np.int64)

train_X, train_y, test_X, test_y = [], [], [], []
summary = []
example_raw = example_clean = example_beats = None

for r in RECORDS:
    raw, clean, X, y = read_beats(r)
    if r == DS1[0]:
        example_raw, example_clean, example_beats = raw, clean, X[:3]
    if r in DS1:
        train_X.append(X); train_y.append(y); split = "Train DS1"
    else:
        test_X.append(X); test_y.append(y); split = "Test DS2"
    summary.append({"record":r, "split":split, "beats":len(y)})
    print(r, len(y), "beats")

X_train = np.concatenate(train_X); y_train = np.concatenate(train_y)
X_test = np.concatenate(test_X); y_test = np.concatenate(test_y)
X_train = X_train[...,None]; X_test = X_test[...,None]
display(pd.DataFrame(summary))
print("Train:", X_train.shape, "Test:", X_test.shape)


## 4. Plot ECG preprocessing

In [ ]:
n = min(len(example_raw), 5*FS)
t = np.arange(n)/FS
plt.figure(figsize=(13,4))
plt.plot(t, example_raw[:n], label="Raw ECG", alpha=.7)
plt.plot(t, example_clean[:n], label="Filtered ECG", linewidth=1)
plt.title(f"Raw vs filtered ECG — record {DS1[0]}")
plt.xlabel("Time (seconds)"); plt.ylabel("Amplitude"); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12,3))
for i, ax in enumerate(axes):
    if example_beats is not None and i < len(example_beats):
        ax.plot(np.arange(SEG_LEN)/FS, example_beats[i], color="seagreen")
        ax.set_title(f"Segmented beat {i+1}"); ax.set_xlabel("Seconds"); ax.grid(alpha=.25)
axes[0].set_ylabel("Normalized amplitude")
plt.tight_layout(); plt.show()


## 5. Inspect class balance

In [ ]:
def counts(y, split):
    return pd.DataFrame({"Class":CLASSES, "Count":np.bincount(y, minlength=len(CLASSES)), "Split":split})
distribution = pd.concat([counts(y_train,"Train"), counts(y_test,"Test")], ignore_index=True)
display(distribution)
distribution.pivot(index="Class",columns="Split",values="Count").plot(kind="bar",figsize=(9,4))
plt.title("Heartbeat class distribution"); plt.ylabel("Number of beats")
plt.xticks(rotation=0); plt.grid(axis="y",alpha=.25); plt.tight_layout(); plt.show()


## 6. Build and train the 1D CNN

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=.15, random_state=SEED, stratify=y_train
)
cw = compute_class_weight("balanced", classes=np.unique(y_fit), y=y_fit)
class_weights = {int(c):float(w) for c,w in zip(np.unique(y_fit),cw)}
print("Class weights:", class_weights)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEG_LEN,1)),
    tf.keras.layers.Conv1D(32,7,padding="same",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(64,5,padding="same",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(96,3,padding="same",activation="relu"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(.35),
    tf.keras.layers.Dense(64,activation="relu"),
    tf.keras.layers.Dropout(.25),
    tf.keras.layers.Dense(len(CLASSES),activation="softmax")
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=2,min_lr=1e-5)
]
history = model.fit(X_fit,y_fit,validation_data=(X_val,y_val),epochs=25,batch_size=256,
                    class_weight=class_weights,callbacks=callbacks,verbose=1)


## 7. Training graphs

In [ ]:
h = history.history
epochs = range(1,len(h["loss"])+1)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(epochs,h["accuracy"],label="Train"); plt.plot(epochs,h["val_accuracy"],label="Validation")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(alpha=.25)
plt.subplot(1,2,2)
plt.plot(epochs,h["loss"],label="Train"); plt.plot(epochs,h["val_loss"],label="Validation")
plt.title("Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.show()


## 8. Final test-set metrics
The DS2 records are held out until this point. Use the metrics printed here as your own prototype results only after the notebook finishes successfully.

In [ ]:
prob = model.predict(X_test,batch_size=512,verbose=1)
y_pred = np.argmax(prob,axis=1)

print("TEST SET METRICS")
print(f"Accuracy:            {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision (macro):   {precision_score(y_test,y_pred,average='macro',zero_division=0):.4f}")
print(f"Recall (macro):      {recall_score(y_test,y_pred,average='macro',zero_division=0):.4f}")
print(f"F1-score (macro):    {f1_score(y_test,y_pred,average='macro',zero_division=0):.4f}")
print(f"F1-score (weighted): {f1_score(y_test,y_pred,average='weighted',zero_division=0):.4f}")
print("\nPer-class report:")
print(classification_report(y_test,y_pred,labels=list(range(len(CLASSES))),
                            target_names=CLASSES,zero_division=0,digits=4))


## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test,y_pred,labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(ax=ax,cmap="Greens",values_format="d",colorbar=False)
ax.set_title("1D CNN — DS2 test confusion matrix")
plt.tight_layout(); plt.show()


## 10. Sample predictions

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.choice(len(X_test),size=min(6,len(X_test)),replace=False)
fig, axes = plt.subplots(len(indices),1,figsize=(10,2*len(indices)))
if len(indices)==1: axes=[axes]
for ax, idx in zip(axes,indices):
    ax.plot(np.arange(SEG_LEN)/FS,X_test[idx,:,0],color="seagreen")
    ax.set_title(f"True: {CLASSES[y_test[idx]]} | Predicted: {CLASSES[y_pred[idx]]}")
    ax.set_xlabel("Time (seconds)"); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


## 11. Save model and metrics

In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test,y_pred),
    "precision_macro": precision_score(y_test,y_pred,average="macro",zero_division=0),
    "recall_macro": recall_score(y_test,y_pred,average="macro",zero_division=0),
    "f1_macro": f1_score(y_test,y_pred,average="macro",zero_division=0),
    "f1_weighted": f1_score(y_test,y_pred,average="weighted",zero_division=0)
}
model.save("/content/ecg_arrhythmia_1d_cnn.keras")
pd.DataFrame([metrics]).to_csv("/content/ecg_test_metrics.csv",index=False)
display(pd.DataFrame([metrics]).T.rename(columns={0:"Score"}))
print("Saved model: /content/ecg_arrhythmia_1d_cnn.keras")
print("Saved metrics: /content/ecg_test_metrics.csv")
